# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [14]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.1886972923742736)),
 (1, np.float64(2.1886972923742736)),
 (2, np.float64(2.1886972923742736)),
 (3, np.float64(2.1886972923742736)),
 (4, np.float64(2.1886972923742736))]

## Inverted index 

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('is', 18), ('it', 18)]),
 (1, [('a', 2), ('what', 10)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.007423964812278916)),
   (None, np.float64(0.038864160776927004)),
   (None, np.float64(0.049809992041082496)),
   (None, np.float64(0.06389576614125114)),
   (None, np.float64(0.0746967357654118)),
   (None, np.float64(0.07620370345524785)),
   (None, np.float64(0.10663129453310483)),
   (None, np.float64(0.12259523314937393)),
   (None, np.float64(0.12490199842133676)),
   (None, np.float64(0.16749367520358205)),
   (None, np.float64(0.2461048761075646)),
   (None, np.float64(0.24936161027808867)),
   (None, np.float64(0.2803431896644917)),
   (None, np.float64(0.3114495768557055)),
   (None, np.float64(0.37457150935567696)),
   (None, np.float64(0.42968311726891684)),
   (None, np.float64(0.44471165262261136)),
   (None, np.float64(0.46157107764245653))]),
 (1,
  [(None, np.float64(0.5269947204696811)),
   (None, np.float64(0.5519785326872272)),
   (None, np.float64(0.6047145347612076)),
   (None, np.float64(0.6179761376233167)),
   (None, np.float64(0.65

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
numbers = [-1, 10, 2, 3, 11, 5]

def RECORDREADER():
    for idx, val in enumerate(numbers):
        yield (idx, val)

def MAP(_, number: int):
    yield ('max', number)

def REDUCE(key, values: Iterator[int]):
    maximum = -np.inf
    for v in values:
        maximum = v if v > maximum else maximum
    yield key, maximum

list(MapReduce(RECORDREADER, MAP, REDUCE))


[('max', 11)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [26]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8]

def RECORDREADER():
    for idx, val in enumerate(numbers):
        yield (idx, val)

def MAP(_, number: int):
    yield ('mean', number)

def REDUCE(key, values: Iterator[int]):
    sum = 0
    count = 0
    for value in values:
        sum += value
        count += 1

    yield (key, sum / count if count > 0 else 0)
    
list(MapReduce(RECORDREADER, MAP, REDUCE))


[('mean', 4.5)]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [27]:
def groupbykey_sorted(iterable: Iterator):
    sorted_items = sorted(iterable, key=lambda x: x[0])

    result = []
    if not sorted_items:
        return result
    
    current_key = sorted_items[0][0]
    current_values = [sorted_items[0][1]]

    for i in range(1, len(sorted_items)):
        key, value = sorted_items[i]
        if key == current_key:
            current_values.append(value)
        else:
            result.append((current_key, current_values))
            current_key = key
            current_values = [value]

    result.append((current_key, current_values))
    
    return result

In [29]:
d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)


def MapReduceSorted(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey_sorted(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

output = MapReduceSorted(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('a', 1), ('banana', 1), ('is', 9), ('it', 9), ('what', 5)]

### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [35]:
maps = 3
reducers = 2

numbers1 = [1, 1, 2]
numbers2 = [3, 4, 5, 6]
numbers3 = [6, 7, 89, 1]
numbers4 = [1, 20, 21]
numbers5 = [31, 4, 51, 61]
numbers6 = [62, 7, 891, 1]


total_numbers = [numbers1, numbers2, numbers3, numbers4, numbers5, numbers6]

def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for (arr_ind, arr) in enumerate(split):
            for (_, num) in enumerate(arr):
                yield (arr_ind, num)
    
    split_size = int(np.ceil(len(total_numbers) / maps))
    for i in range(0, len(total_numbers), split_size):
        yield RECORDREADER(total_numbers[i:i + split_size])

def MAP(_, num):
    yield (num, None)

def REDUCE(key, values):
    yield (key, None)

def COMBINER(key, values):
    yield (key, None)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=COMBINER) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

20 key-value pairs were sent over a network.


[(0, [(2, None), (4, None), (6, None), (20, None), (62, None)]),
 (1,
  [(1, None),
   (3, None),
   (5, None),
   (7, None),
   (21, None),
   (31, None),
   (51, None),
   (61, None),
   (89, None),
   (891, None)])]

# Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
R = [
    (1, 25, 'F'),
    (2, 30, 'M'),
    (3, 25, 'M'),
    (4, 30, 'F'),
]

def predicate(value):
    return value[1] > 25

def RECORDREADER():
    for idx, t in enumerate(R):
        yield (idx, t)

def MAP(_, t):
    if predicate(t):
        yield (t, t)

def REDUCE(key, _):
    yield key

list(MapReduce(RECORDREADER, MAP, REDUCE))

[(2, 30, 'M'), (4, 30, 'F')]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [44]:
R = [
    (1, 25, 'F'),
    (2, 30, 'M'),
    (3, 25, 'M'),
    (4, 30, 'F'),
]

def project(value):
    return (value[0], value[2])

def RECORDREADER():
    for idx, t in enumerate(R):
        yield (idx, t)

def MAP(_, t):
    t_ = project(t)
    yield (t_, t_)

def REDUCE(key, value):
    yield (key, key)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[((1, 'F'), (1, 'F')),
 ((2, 'M'), (2, 'M')),
 ((3, 'M'), (3, 'M')),
 ((4, 'F'), (4, 'F'))]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [50]:
R = [
    (1, 25, 'F'),
    (2, 30, 'M'),
    (3, 25, 'M'),
    (4, 30, 'F'),
]

S = [
    (3, 25, 'M'),
    (4, 30, 'F'),
    (5, 35, 'F'),
]


def RECORDREADER():
    for idx, t in enumerate(R):
        yield (idx, t)
    for idx, t in enumerate(S):
        yield (idx + len(R), t)

def MAP(_, t):
    yield (t, t)

def REDUCE(key, values: Iterator):
    yield (key, key)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[((1, 25, 'F'), (1, 25, 'F')),
 ((2, 30, 'M'), (2, 30, 'M')),
 ((3, 25, 'M'), (3, 25, 'M')),
 ((4, 30, 'F'), (4, 30, 'F')),
 ((5, 35, 'F'), (5, 35, 'F'))]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [52]:
R = [
    (1, 25, 'F'),
    (2, 30, 'M'),
    (3, 25, 'M'),
    (4, 30, 'F'),
]

S = [
    (3, 25, 'M'),
    (4, 30, 'F'),
    (5, 35, 'F'),
]

def RECORDREADER():
    for idx, t in enumerate(R):
        yield (idx, t)
    for idx, t in enumerate(S):
        yield (idx + len(R), t)

def MAP(_, t):
    yield (t, t)

def REDUCE(key, values: Iterator):
    count = 0
    for _ in values:
        count += 1

    if count == 2:
        yield (key, key)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[((3, 25, 'M'), (3, 25, 'M')), ((4, 30, 'F'), (4, 30, 'F'))]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [53]:
R = [
    (1, 25, 'F'),
    (2, 30, 'M'),
    (3, 25, 'M'),
    (4, 30, 'F'),
]

S = [
    (3, 25, 'M'),
    (4, 30, 'F'),
    (5, 35, 'F'),
]

def RECORDREADER():
    for idx, t in enumerate(R):
        yield (idx, (t, 'R'))
    for idx, t in enumerate(S):
        yield (idx, (t, 'S'))

def MAP(key, value):
    yield value

def REDUCE(key, values: Iterator):
    sources = []
    for source in values:
        sources.append(source)
    
    if sources == ['R']:
        yield (key, key)

list(MapReduce(RECORDREADER, MAP, REDUCE))

[((1, 25, 'F'), (1, 25, 'F')), ((2, 30, 'M'), (2, 30, 'M'))]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [67]:
R = [
    (1, 25),
    (2, 30),
    (3, 25),
    (4, 30),
]

S = [
    (25, 'M'),
    (30, 'F'),
    (35, 'F'),
]

def RECORDREADER():
    for idx, t in enumerate(R):
        yield (idx, (t, 'R'))
    for idx, t in enumerate(S):
        yield (idx + len(R), (t, 'S'))

def MAP(key, value):
    t, source = value
    
    if source == 'R':
        a, b = t
        yield (b, ('R', a))
    else:
        b, c = t
        yield (b, ('S', c))

def REDUCE(key, values: Iterator):
    r_values = []
    s_values = []
    
    for source, attr in values:
        if source == 'R':
            r_values.append(attr)
        else:
            s_values.append(attr)
    
    for a in r_values:
        for c in s_values:
            yield (a, key, c)
    

list(MapReduce(RECORDREADER, MAP, REDUCE))

[(1, 25, 'M'), (3, 25, 'M'), (2, 30, 'F'), (4, 30, 'F')]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [68]:
R = [
    (1, 10, 'X'),
    (1, 20, 'Y'),
    (1, 30, 'Z'),
    (2, 5, 'A'),
    (2, 15, 'B'),
    (3, 100, 'C'),
]

def RECORDREADER():
    for idx, t in enumerate(R):
        yield (idx, t)

def MAP(key, t):
    yield (t[0], t[1])

def REDUCE(key: int, values: Iterator[int]):
    max_result = None
    for v in values:
        if max_result is None or v > max_result:
            max_result = v
    yield (key, max_result)
    
list(MapReduce(RECORDREADER, MAP, REDUCE))

[(1, 30), (2, 15), (3, 100)]

# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [73]:
I, J = 3, 4
small_mat = np.ones((I, J))
vector = np.random.rand(J)


def RECORDREADER1():
    for i in range(I):
        for j in range(J):
            yield ((i, j), ('M', small_mat[i, j]))
    for j in range(J):
        yield (j, ('V', vector[j]))

def MAP1(key, value):
    if isinstance(key, tuple):
        i, j = key
        _, m_ij = value
        yield (j, ('M', i, m_ij))
    else:
        j = key
        _, v_j = value
        yield (j, ('V', v_j))

def REDUCE1(j: int, values):
    matrix_elements = []
    v_j = None
    for item in values:
        if item[0] == 'M':
            _, i, m_ij = item
            matrix_elements.append((i, m_ij))
        else:
            _, v_j = item
    if v_j is not None:
        for i, m_ij in matrix_elements:
            yield (i, m_ij * v_j)

def RECORDREADER2(partial_results):
    for i, partial in partial_results:
        yield (i, partial)

def MAP2(i, partial):
    yield (i, partial)

def REDUCE2(i: int, values: Iterator[float]):
    total = sum(values)
    yield (i, total)

partial = list(MapReduce(RECORDREADER1, MAP1, REDUCE1))
list(MapReduce(lambda: RECORDREADER2(partial), MAP2, REDUCE2))

[(0, np.float64(2.5572630628535387)),
 (1, np.float64(2.5572630628535387)),
 (2, np.float64(2.5572630628535387))]

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [23]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [74]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j, k), big_mat[j, k])

def MAP(k1, v1):
    (j, k) = k1
    w = v1
    
    for i in range(small_mat.shape[0]):
        m_ij = small_mat[i, j]
        yield ((i, k), m_ij * w)

def REDUCE(key, values):
    (i, k) = key
    total = 0
    for v in values:
        total += v
    yield (key, total)

Проверьте своё решение

In [75]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [76]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [79]:
import numpy as np

I, J, K = 2, 3, 4
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

#
# двухпроходный алгоритм
# 1 - джоин по j
# 2 - джоин по (i, k)

def RECORDREADER1():
    for i in range(I):
        for j in range(J):
            yield ((i, j), ('M', mat_M[i, j]))
    
    for j in range(J):
        for k in range(K):
            yield ((j, k), ('N', mat_N[j, k]))

def MAP1(key, value):
    if value[0] == 'M':
        i, j = key
        _, m_ij = value
        yield (j, ('M', i, m_ij))
    else:
        j, k = key
        _, n_jk = value
        yield (j, ('N', k, n_jk))

def REDUCE1(j, values):
    m_elements = []
    n_elements = []
    
    for item in values:
        if item[0] == 'M':
            _, i, m_ij = item
            m_elements.append((i, m_ij))
        else:
            _, k, n_jk = item
            n_elements.append((k, n_jk))
    
    for (i, m_ij) in m_elements:
        for (k, n_jk) in n_elements:
            yield ((i, k), m_ij * n_jk)


def RECORDREADER2(partial_results):
    for ik, product in partial_results:
        yield (ik, product)

def MAP2(ik, product):
    yield (ik, product) 

def REDUCE2(ik, products):
    total = 0
    for p in products:
        total += p
    yield (ik, total)

partial_results = list(MapReduce(RECORDREADER1, MAP1, REDUCE1))

final_results = list(MapReduce(lambda: RECORDREADER2(partial_results), MAP2, REDUCE2))

def asmatrix(reduce_output):
    reduce_output = list(reduce_output)
    I = max(i for ((i, k), v) in reduce_output) + 1
    K = max(k for ((i, k), v) in reduce_output) + 1
    mat = np.empty(shape=(I, K))
    for ((i, k), v) in reduce_output:
        mat[i, k] = v
    return mat

reference_solution = np.matmul(mat_M, mat_N)
result_matrix = asmatrix(final_results)

print(result_matrix)
np.allclose(reference_solution, result_matrix)

[[0.94889929 0.7793994  0.9536215  0.81658598]
 [0.93931331 0.7531319  0.97626428 0.7595921 ]]


True

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [94]:
I, J, K = 4, 3, 5
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

maps = 2
reducers = 2

# первый проход: джоин по j

def INPUTFORMAT1():
    global maps
    
    def RECORDREADER_M(split):
        for i, j in split:
            yield ((i, j), ('M', mat_M[i, j]))
    
    def RECORDREADER_N(split):
        for j, k in split:
            yield ((j, k), ('N', mat_N[j, k]))
    
    m_elements = [(i, j) for i in range(I) for j in range(J)]
    split_size_m = max(1, int(np.ceil(len(m_elements) / maps)))
    for i in range(0, len(m_elements), split_size_m):
        yield RECORDREADER_M(m_elements[i:i + split_size_m])
    
    n_elements = [(j, k) for j in range(J) for k in range(K)]
    split_size_n = max(1, int(np.ceil(len(n_elements) / maps)))
    for i in range(0, len(n_elements), split_size_n):
        yield RECORDREADER_N(n_elements[i:i + split_size_n])

def MAP1(key, value):
    if value[0] == 'M':
        i, j = key
        _, m_ij = value
        yield (j, ('M', i, m_ij))
    else:
        j, k = key
        _, n_jk = value
        yield (j, ('N', k, n_jk))

def REDUCE1(j, values: Iterator):
    m_elements = []
    n_elements = []
    
    for item in values:
        if item[0] == 'M':
            _, i, m_ij = item
            m_elements.append((i, m_ij))
        else:
            _, k, n_jk = item
            n_elements.append((k, n_jk))
    
    if len(m_elements) > 0 and len(n_elements) > 0:
        for (i, m_ij) in m_elements:
            for (k, n_jk) in n_elements:
                yield ((i, k), m_ij * n_jk)

# втоорой проход: суммирование по (i, k)

def INPUTFORMAT2(partial_results):
    global maps
    
    def RECORDREADER(split):
        for ik, product in split:
            yield (ik, product)
    
    if len(partial_results) == 0:
        return
    split_size = max(1, int(np.ceil(len(partial_results) / maps)))
    for i in range(0, len(partial_results), split_size):
        yield RECORDREADER(partial_results[i:i + split_size])

def MAP2(ik, product):
    yield (ik, product)

def REDUCE2(ik, products: Iterator[float]):
    total = 0
    for p in products:
        total += p
    yield (ik, total)

def COMBINER2(ik, products: Iterator[float]):
    total = 0
    for p in products:
        total += p
    yield (ik, total)

partial_results = MapReduceDistributed(
    INPUTFORMAT1, 
    MAP1, 
    REDUCE1, 
    COMBINER=None
)
partial_results = [(partition_id, list(partition)) for (partition_id, partition) in partial_results]

partial_flat = []
for (partition_id, partition) in partial_results:
    for (ik, product) in partition:
        partial_flat.append((ik, product))

final_results = MapReduceDistributed(
    lambda: INPUTFORMAT2(partial_flat),
    MAP2, 
    REDUCE2, 
    COMBINER=COMBINER2
)
final_results = [(partition_id, list(partition)) for (partition_id, partition) in final_results]

result_dict = {}
for (partition_id, partition) in final_results:
    for (ik, value) in partition:
        result_dict[ik] = value

def asmatrix(result_dict, I, K):
    mat = np.empty(shape=(I, K))
    for i in range(I):
        for k in range(K):
            mat[i, k] = result_dict.get((i, k), 0)
    return mat

result_matrix = asmatrix(result_dict, I, K)
reference_solution = np.matmul(mat_M, mat_N)

print(result_matrix)
np.allclose(reference_solution, result_matrix)

27 key-value pairs were sent over a network.
40 key-value pairs were sent over a network.
[[0.92148864 0.30886956 0.08933296 0.81533294 0.60248085]
 [0.977461   0.32288448 0.09960218 0.8407903  0.59609309]
 [1.11959715 0.34798983 0.22568165 0.75905847 0.26414848]
 [1.42301834 0.46660045 0.22040139 1.13146958 0.65877577]]


True

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [97]:
I, J, K = 4, 3, 5
mat_M = np.random.rand(I, J)
mat_N = np.random.rand(J, K)

maps = 3
reducers = 2

# первый ароход: джоин по j

def INPUTFORMAT1():
    global maps
    
    def RECORDREADER_M(split, sparsity=1.0):
        for i, j in split:
            if np.random.rand() < sparsity:
                yield ((i, j), ('M', mat_M[i, j]))
    
    def RECORDREADER_N(split, sparsity=1.0):
        for j, k in split:
            if np.random.rand() < sparsity:
                yield ((j, k), ('N', mat_N[j, k]))
    
    m_elements = [(i, j) for i in range(I) for j in range(J)]
    split_size_m = max(1, int(np.ceil(len(m_elements) / maps)))
    for i in range(0, len(m_elements), split_size_m):
        yield RECORDREADER_M(m_elements[i:i + split_size_m], sparsity=1.0)
    
    n_elements = [(j, k) for j in range(J) for k in range(K)]
    split_size_n = max(1, int(np.ceil(len(n_elements) / maps)))
    for i in range(0, len(n_elements), split_size_n):
        yield RECORDREADER_N(n_elements[i:i + split_size_n], sparsity=1.0)

def MAP1(key, value):
    if value[0] == 'M':
        i, j = key
        _, m_ij = value
        yield (j, ('M', i, m_ij))
    else:
        j, k = key
        _, n_jk = value
        yield (j, ('N', k, n_jk))

def REDUCE1(j, values: Iterator):
    m_elements = []
    n_elements = []
    
    for item in values:
        if item[0] == 'M':
            _, i, m_ij = item
            m_elements.append((i, m_ij))
        else:
            _, k, n_jk = item
            n_elements.append((k, n_jk))
    
    if len(m_elements) > 0 and len(n_elements) > 0:
        for (i, m_ij) in m_elements:
            for (k, n_jk) in n_elements:
                yield ((i, k), m_ij * n_jk)

# второй проход: суммирование по (i, k)

def INPUTFORMAT2(partial_results):
    global maps
    
    def RECORDREADER(split):
        for ik, product in split:
            yield (ik, product)
    
    if len(partial_results) == 0:
        return
    split_size = max(1, int(np.ceil(len(partial_results) / maps)))
    for i in range(0, len(partial_results), split_size):
        yield RECORDREADER(partial_results[i:i + split_size])

def MAP2(ik, product):
    yield (ik, product)

def REDUCE2(ik, products: Iterator[float]):
    total = 0
    for p in products:
        total += p
    yield (ik, total)

def COMBINER2(ik, products: Iterator[float]):
    total = 0
    for p in products:
        total += p
    yield (ik, total)

partial_results = MapReduceDistributed(
    INPUTFORMAT1, 
    MAP1, 
    REDUCE1, 
    COMBINER=None
)
partial_results = [(partition_id, list(partition)) for (partition_id, partition) in partial_results]

partial_flat = []
for (partition_id, partition) in partial_results:
    for (ik, product) in partition:
        partial_flat.append((ik, product))

final_results = MapReduceDistributed(
    lambda: INPUTFORMAT2(partial_flat),
    MAP2, 
    REDUCE2, 
    COMBINER=COMBINER2
)
final_results = [(partition_id, list(partition)) for (partition_id, partition) in final_results]

result_dict = {}
for (partition_id, partition) in final_results:
    for (ik, value) in partition:
        result_dict[ik] = value

def asmatrix(result_dict, I, K):
    mat = np.zeros(shape=(I, K))
    for i in range(I):
        for k in range(K):
            mat[i, k] = result_dict.get((i, k), 0)
    return mat

result_matrix = asmatrix(result_dict, I, K)
reference_solution = np.matmul(mat_M, mat_N)

print(result_matrix)
np.allclose(reference_solution, result_matrix)

27 key-value pairs were sent over a network.
60 key-value pairs were sent over a network.
[[1.44189628 0.81648312 1.18138925 1.47788158 1.17682982]
 [1.3487998  0.97060132 1.32544009 1.55798704 1.01911795]
 [0.92152206 0.57597513 0.88954636 0.9516714  0.66115785]
 [1.1428613  0.73268347 0.96311497 1.2764871  0.95725772]]


True